In [1]:
import tensorflow as tf
from tensorflow import keras
import numpy as np

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
base_image_path = keras.utils.get_file(
    "sf.jpg",
    origin="https://img-datasets.s3.amazonaws.com/sf.jpg"
)

style_reference_image_path_1 = keras.utils.get_file(
    "starry_night.jpg", origin="https://img-datasets.s3.amazonaws.com/starry_night.jpg"
)

style_reference_image_path_2 = keras.utils.get_file(
    "sunflowers.jpg", origin="https://upload.wikimedia.org/wikipedia/commons/thumb/4/46/Vincent_Willem_van_Gogh_127.jpg/800px-Vincent_Willem_van_Gogh_127.jpg"
)

original_width, original_height = keras.utils.load_img(base_image_path).size
img_height = 400
img_width = round(original_width * img_height / original_height)


In [ ]:
def preprocess_image(image_path):
      img = keras.utils.load_img(
      image_path, target_size=(img_height, img_width))
      img = keras.utils.img_to_array(img)
      img = np.expand_dims(img, axis=0)
      img = keras.applications.vgg19.preprocess_input(img)
      return img

In [ ]:
def deprocess_image(img):
    img = img.reshape((img_height, img_width, 3))
    img[:, :, 0] += 103.939
    img[:, :, 1] += 116.779
    img[:, :, 2] += 123.68
    img = img[:, :, ::-1]
    img = np.clip(img, 0, 255).astype("uint8")
    return img

In [ ]:
model = keras.applications.vgg19.VGG19(weights="imagenet", include_top=False)
outputs_dict = dict([(layer.name, layer.output) for layer in model.layers])
feature_extractor = keras.Model(inputs=model.inputs, outputs=outputs_dict)


In [ ]:
def content_loss(base_img, combination_img):
    return tf.reduce_sum(tf.square(combination_img - base_img))


In [ ]:
def gram_matrix(x):
    x = tf.transpose(x, (2, 0, 1))
    features = tf.reshape(x, (tf.shape(x)[0], -1))
    gram = tf.matmul(features, tf.transpose(features))
    return gram


In [ ]:

def style_loss(style_img, combination_img):
    S = gram_matrix(style_img)
    C = gram_matrix(combination_img)
    channels = 3
    size = img_height * img_width
    return tf.reduce_sum(tf.square(S - C)) / (4.0 * (channels ** 2) * (size ** 2))


In [ ]:
def total_variation_loss(x):
    a = tf.square(
    x[:, : img_height - 1, : img_width - 1, :] - x[:, 1:, : img_width - 1, :]
    )
    b = tf.square(
    x[:, : img_height - 1, : img_width - 1, :] - x[:, : img_height - 1, 1:, :]
    )
    return tf.reduce_sum(tf.pow(a + b, 1.25))


In [ ]:
content_layer_names = [
    "block1_conv1", 
    "block5_conv2"]


style_layer_names = [
    "block1_conv1",
    "block2_conv1",
    "block3_conv1",
    "block4_conv1",
    "block5_conv1"]


In [ ]:
def blended_style_loss(style_features_1, style_features_2, combination_features):
    S1 = gram_matrix(style_features_1)
    S2 = gram_matrix(style_features_2)
    S_blend = 0.5 * S1 + 0.5 * S2 
    C = gram_matrix(combination_features)
    channels = 3
    size = img_height * img_width
    return tf.reduce_sum(tf.square(S_blend - C)) / (4.0 * (channels ** 2) * (size ** 2))

In [ ]:
content_layer_name = "block5_conv2"
total_variation_weight = 1e-6

style_weight = 1e-6
content_weight = 2.5e-8


In [ ]:
def compute_loss(combination_image, base_image, style_image_1, style_image_2):
    input_tensor = tf.concat(
        [base_image, style_image_1, style_image_2, combination_image], axis=0
    )
    features = feature_extractor(input_tensor)
    loss = tf.zeros(shape=())

    for layer_name in content_layer_names:
        layer_features = features[layer_name]
        base_image_features = layer_features[0, :, :, :] 
        combination_features = layer_features[3, :, :, :] 
        loss = loss + (content_weight / len(content_layer_names)) * content_loss(
            base_image_features, combination_features
        )

    for layer_name in style_layer_names:
        layer_features = features[layer_name]
        style_features_1 = layer_features[1, :, :, :]
        style_features_2 = layer_features[2, :, :, :] 
        combination_features = layer_features[3, :, :, :]
        
        style_loss_value = blended_style_loss(
            style_features_1, style_features_2, combination_features
        )
        loss += (style_weight / len(style_layer_names)) * style_loss_value

    loss += total_variation_weight * total_variation_loss(combination_image)
    return loss

In [ ]:
@tf.function
def compute_loss_and_grads(combination_img, base_img, style_img_1, style_img_2):
    with tf.GradientTape() as tape:
        loss = compute_loss(combination_img, base_img, style_img_1, style_img_2)
    grads = tape.gradient(loss, combination_img)
    return loss, grads
optimizer = keras.optimizers.SGD(keras.optimizers.schedules.ExponentialDecay(initial_learning_rate=100.0, decay_steps=100, decay_rate=0.96))


In [ ]:
base_image = preprocess_image(base_image_path)
style_reference_image_1 = preprocess_image(style_reference_image_path_1)
style_reference_image_2 = preprocess_image(style_reference_image_path_2)
combination_image = tf.Variable(preprocess_image(base_image_path))

iterations = 4000
for i in range(1, iterations + 1):
    loss, grads = compute_loss_and_grads(
        combination_image, base_image, style_reference_image_1, style_reference_image_2
    )
    optimizer.apply_gradients([(grads, combination_image)])
    
    if i % 100 == 0:
        print(f"Iteracja {i}: loss={loss:.2f}")
        img = deprocess_image(combination_image.numpy())
        fname = f"eksperyment_kolor_styl_at_iteration_{i}.png"
        keras.utils.save_img(fname, img)